# Verne, small, step 07: grasp lower (a data step), on T1-wide

No explained notebook: a data step plus training changes. Step 06: `06_dagger.ipynb`. T5 (same day): `08_t5_instruction.ipynb`.

**Why:** the expert aimed its grip point 0.5 cm above the cube centre and starts closing within 1 cm of that target while still descending, so the fingers caught the cube's top quarter (+1.5 cm above centre on average, half-height ≈ 2 cm). Under joint noise σ 0.05 (the full shake), measured on T5 (`experiments/verne/scripts/grasp_height_t5.py`, 180 seeds):

```-
target         starts closing within   placed     close height (mean)
+0.5 cm (old)  1.0 cm                  150/180    +1.5 cm
-0.5 cm (new)  1.0 cm                  163/180    +0.6 cm
-0.5 cm        0.5 cm                  160/180    +0.5 cm
 0             0.5 cm                  143/180    +1.0 cm
```

`grasp_dz` is now −0.005 in both task.yamls (T1 and T5); without shake both experts place 50/50. New T1-wide data: `pick_place_cube_wide_square_low_noise005_n100` (square up + full shake + low grasp), expert 93% (100 of 107; the high-grasp set: 85%, 100 of 118).

**What else changed in the package since step 06** (all runs below except step 05/06 use it):
- the VAE: retrained on T1, T1-wide low, T1-wide ±45° bias and T5 frames (for T5's colours, see `08_t5_instruction.ipynb`); reconstruction on low-grasp frames 0.0015 (step 03 VAE: 0.0037);
- the instruction one-hot as a video token and added to the action tokens (all zeros on T1);
- gradient clipping at norm 1.0, checkpoints every 2,000 steps;
- the LR schedule: first warmup-stable-decay (WSD), then back to cosine (step 07.2).

Seed 0, 50 rollout seeds at n_exec 8 and 16; the new runs trained on Modal (L40S).

In [ ]:
import json

from dream_robot.policies.verne.run import RUNS

RUNS_07 = {
    'step 05: square + shake (high grasp)': RUNS / '05_square' / 't1-wide-square-noise' / 'seed0',
    'step 06: + DAgger r1 + r2 (failed)': RUNS
    / '06_dagger'
    / 't1-wide-square-noise+dagger_r1:fail+dagger_r2:fail'
    / 'seed0',
    'step 07: low grasp, WSD': RUNS / '07_instruction_act' / 't1-wide-low' / 'seed0',
    'step 07: low grasp, cosine': RUNS / '07_cosine' / 't1-wide-low' / 'seed0',
}

In [ ]:
def read_run(path):
    # a run folder -> n_exec -> (successes, failure counts), probe fit, final val loss
    out = {}
    for n in (8, 16):
        res = json.loads((path / f'sweep_{n}.json').read_text()).values()
        fails = {}
        for ok, mode, _ in res:
            if not ok:
                fails[mode] = fails.get(mode, 0) + 1
        out[n] = (sum(r[0] for r in res), fails)
    log = json.loads((path / 'log.json').read_text())
    return out, json.loads((path / 'probe.json').read_text())['fit'], log[-1][2]

### step 07.1 — scores

In [ ]:
print(f'{"":38s} {"n_exec 8":>9s} {"n_exec 16":>10s} {"miss":>7s} {"x slope":>8s} {"y slope":>8s}')
for name, path in RUNS_07.items():
    r, fit, val = read_run(path)
    print(
        f'{name:38s} {r[8][0]:6d}/50 {r[16][0]:7d}/50 {fit["miss_cm"]:5.1f} cm '
        f'{fit["slope_x"]:+8.2f} {fit["slope_y"]:+8.2f}   {r[8][1]} | {r[16][1]}'
    )

                                        n_exec 8  n_exec 16    miss  x slope  y slope
step 05: square + shake (high grasp)       42/50      44/50   1.0 cm    +0.94    +0.86   {'no_grasp': 6, 'wrong_target': 1, 'dropped': 1} | {'no_grasp': 6}
step 06: + DAgger r1 + r2 (failed)         48/50      47/50   0.9 cm    +1.02    +0.97   {'wrong_target': 2} | {'dropped': 2, 'no_grasp': 1}
step 07: low grasp, WSD                    29/50      23/50   3.1 cm    +0.61    +0.69   {'no_grasp': 21} | {'no_grasp': 27}
step 07: low grasp, cosine                 43/50      33/50   1.5 cm    +0.86    +0.84   {'no_grasp': 7} | {'no_grasp': 16, 'wrong_target': 1}


**WSD dropped T1-wide to 29/23; back on cosine it's 43/33** — same data, same VAE, same model, only the schedule differs.
- WSD (warmup 1k, flat 3e-4 to step 16k, decay to 0 by 20k; mean lr ~2.6e-4) found the cube less precisely: close point 3.1 cm off, slopes 0.61 / 0.69. Cosine (3e-4 → 0 from step 0, as steps 02-06; mean lr ~1.5e-4): 1.5 cm, 0.86 / 0.84.
- WSD fit the training demos harder (final train loss 0.014) but generalised worse to new cube spots. Verne's training is back on cosine (`lr.cosine`); the VAE keeps WSD (it stopped its loss spikes, see `08_t5_instruction.ipynb`).
- Cosine at n_exec 8 matches step 05 (43 vs 42); at n_exec 16 it's lower (33 vs 44). Still four differences from step 05 (low grasp, new VAE, clipping, the action-side input) and one seed, so the n_exec 16 gap isn't attributed yet.
- Failures are `no_grasp` (one `wrong_target` at n_exec 16 with cosine).

### step 07.2 — the held-out loss is not usable here

Both new runs end with held-out loss ~1.1 (step 05: 0.096) while training loss is lower than ever. One held-out demo, episode 82, has an action ~10 standard deviations off the training mean (the worst in the high-grasp set was 8, and in a training episode). With 10 held-out demos the fixed validation batches are dominated by it. Rollouts are the metric.

### step 07.3 — what the failures look like (cosine run, n_exec 8)

- successes: [seed 1000](../../scratch/verne07/t5_seed1000_cube_the_placed.mp4), [seed 1001](../../scratch/verne07/t5_seed1001_cube_the_placed.mp4)
- failures: [seed 1004](../../scratch/verne07/t5_seed1004_cube_the_no_grasp.mp4), [seed 1009](../../scratch/verne07/t5_seed1009_cube_the_no_grasp.mp4), [seed 1022](../../scratch/verne07/t5_seed1022_cube_the_no_grasp.mp4)

Each failure descends to the cube's area and closes just beside it (in 1009 the fingers knock the cube and turn it), then carries an empty gripper to the bin and parks there: the step 04-05 failure, which DAgger (step 06) fixed by labelling exactly these states. (Film names say t5_: the T5 filming script was reused, `experiments/verne/scripts/t5_films.py`.)

### wrap-up — the lower grasp is fine; the schedule was the drop

**Buys:** the expert itself fails less under shake (93% vs 85% on T1-wide, 91% vs 83% on T5), and with cosine the policy is back at step 05's level at n_exec 8 (43/50).

**Fails:** WSD (a long flat phase at full lr) cost T1-wide most of its cube-finding: 29/23. Still open: the n_exec 16 gap (33 vs 44), on one seed.

**Next:** seeds 1-2 on cosine; DAgger on the low-grasp data (step 06's fix for the misses seen here).